In [1]:
from safetensors.torch import load_file
import torch

In [2]:
ckpt_weights = torch.load("../pretrained_weights/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth", map_location='cpu')

In [3]:
model_dict = ckpt_weights['model']
args_dict = ckpt_weights['args']

In [6]:
print(args_dict)
''' 
AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, 
                 dec_num_heads=12, pos_embed='RoPE100',img_size=(512, 512), head_type='catmlp+dpt', 
                 output_mode='pts3d+desc24', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), 
                 patch_embed_cls='PatchEmbedDust3R', two_confs=True, desc_conf_mode=('exp', 0, inf))
'''

Namespace(model="AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, dec_num_heads=12, pos_embed='RoPE100',img_size=(512, 512), head_type='catmlp+dpt', output_mode='pts3d+desc24', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), patch_embed_cls='PatchEmbedDust3R', two_confs=True, desc_conf_mode=('exp', 0, inf))")


In [9]:
# keys = list(model_dict.keys())

# with open("keys.txt", "w") as f:
#     for key in keys:
#         f.write(key + "\n")

In [11]:
# print(len(keys))

1017


In [4]:
fast3r_weights = load_file("../pretrained_weights/Fast3R_ViT_Large_512/model.safetensors")

In [12]:
# fast3r_keys = list(fast3r_weights.keys())
# print(len(fast3r_keys))

# with open("fast3r_keys.txt", "w") as f:
#     for key in fast3r_keys:
#         f.write(key + "\n")

712


In [5]:
def fast3r_checkpoint_filter_fn(fast3r_state_dict, nopo_state_dict):
    """ convert patch embedding weight from manual patchify + linear proj to conv"""
    fast3r_out_dict = {}
    nopo_enc_dict = {}
    # state_dict = state_dict.get('model', state_dict)
    # state_dict = state_dict.get('state_dict', state_dict)

    for k, v in fast3r_state_dict.items():
        if 'encoder.enc_blocks' in k:
            fast3r_out_dict[k] = v
            if k[8:] not in nopo_state_dict:
                nopo_enc_dict[k[8:]] = v

    
    if len(nopo_enc_dict) != 0:
        print("there are different weight names! check by p nopo_enc_dict")
        import pdb; pdb.set_trace()


    for k3, v3 in fast3r_out_dict.items():
        key = k3[8:]
        nopo_state_dict[key] = v3
    # add prefix to make our model happy

    return nopo_state_dict

In [6]:
nopo_dict = fast3r_checkpoint_filter_fn(fast3r_weights, model_dict)

In [8]:
out_dict= {}
out_dict['model'] = nopo_dict
out_dict['args'] = args_dict

In [9]:
torch.save(out_dict, "../pretrained_weights/Fast3R_MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth")

In [19]:
# nopo_keys = list(nopo_dict.keys())

# with open("nopo_keys.txt", "w") as f:
#     for key in nopo_keys:
#         f.write(key + "\n")